In [89]:
from sqlalchemy import create_engine
import pandas as pd

In [90]:
engine = create_engine('sqlite:///sqlite_db_pythonsqlite.db')

In [91]:
query = "SELECT * FROM Bookings;"
result_df = pd.read_sql(query, con = engine)
print(result_df)

      bookid  facid  memid            starttime  slots
0          0      3      1  2012-07-03 11:00:00      2
1          1      4      1  2012-07-03 08:00:00      2
2          2      6      0  2012-07-03 18:00:00      2
3          3      7      1  2012-07-03 19:00:00      2
4          4      8      1  2012-07-03 10:00:00      1
...      ...    ...    ...                  ...    ...
4038    4038      8     29  2012-09-30 16:30:00      2
4039    4039      8     29  2012-09-30 18:00:00      1
4040    4040      8     21  2012-09-30 18:30:00      1
4041    4041      8     16  2012-09-30 19:00:00      1
4042    4042      8     29  2012-09-30 19:30:00      1

[4043 rows x 5 columns]


In [92]:
query = "SELECT * FROM Facilities;"
result_df = pd.read_sql(query, con = engine)
print(result_df)

   facid             name  membercost  guestcost  initialoutlay  \
0      0   Tennis Court 1         5.0       25.0          10000   
1      1   Tennis Court 2         5.0       25.0           8000   
2      2  Badminton Court         0.0       15.5           4000   
3      3     Table Tennis         0.0        5.0            320   
4      4   Massage Room 1         9.9       80.0           4000   
5      5   Massage Room 2         9.9       80.0           4000   
6      6     Squash Court         3.5       17.5           5000   
7      7    Snooker Table         0.0        5.0            450   
8      8       Pool Table         0.0        5.0            400   

   monthlymaintenance  
0                 200  
1                 200  
2                  50  
3                  10  
4                3000  
5                3000  
6                  80  
7                  15  
8                  15  


In [107]:
query = "SELECT surname, firstname, recommendedby FROM Members LIMIT 10;"
result_df = pd.read_sql(query, con = engine)
print(result_df)

    surname firstname recommendedby
0     GUEST     GUEST              
1     Smith    Darren              
2     Smith     Tracy              
3    Rownam       Tim              
4  Joplette    Janice             1
5   Butters    Gerald             1
6     Tracy    Burton              
7      Dare     Nancy             4
8    Boothe       Tim             3
9  Stibbons    Ponder             6


In [94]:
Q10_query = """WITH cte1 AS 
(SELECT f.name,
CASE b.memid
    WHEN 0 THEN f.guestcost
    ELSE f.membercost
END AS revenue
FROM Bookings AS b
JOIN Facilities AS f ON b.facid = f.facid
),

cte2 AS 
(SELECT name as facilityname, SUM(revenue) as totalrevenue
FROM cte1
GROUP BY name
ORDER BY totalrevenue)

SELECT * FROM cte2 WHERE totalrevenue < 1000;"""

result_df = pd.read_sql(Q10_query, con = engine)
print(result_df)

      facilityname  totalrevenue
0     Table Tennis          90.0
1    Snooker Table         115.0
2       Pool Table         265.0
3  Badminton Court         604.5


In [123]:
Q11_query = """SELECT Members.surname, Members.firstname, Recommenders.surname AS recommendersurname, Recommenders.firstname AS recommenderfirstname
FROM Members JOIN Members as Recommenders on Members.recommendedby = Recommenders.memid
ORDER BY Members.surname, Members.firstname;"""

result_df = pd.read_sql(Q11_query, con = engine)
print(result_df)

              surname  firstname recommendersurname recommenderfirstname
0               Bader   Florence           Stibbons               Ponder
1               Baker       Anne           Stibbons               Ponder
2               Baker    Timothy            Farrell               Jemima
3              Boothe        Tim             Rownam                  Tim
4             Butters     Gerald              Smith               Darren
5              Coplin       Joan              Baker              Timothy
6             Crumpet      Erica              Smith                Tracy
7                Dare      Nancy           Joplette               Janice
8             Genting    Matthew            Butters               Gerald
9                Hunt       John            Purview            Millicent
10              Jones      David           Joplette               Janice
11              Jones    Douglas              Jones                David
12           Joplette     Janice              Smith

In [144]:
#Q12_query = "SELECT facid, memid FROM Bookings WHERE memid != 0;"

Q12_query = """WITH cte AS (SELECT Bookings.facid, Bookings.memid FROM Bookings WHERE memid != 0)
SELECT Facilities.name, cte.facid, cte.memid, Members.surname, Members.firstname FROM cte JOIN Facilities ON cte.facid = Facilities.facid JOIN Members ON cte.memid = Members.memid;"""

result_df = pd.read_sql(Q12_query, con = engine)
print(result_df)

                name  facid  memid            surname firstname
0       Table Tennis      3      1              Smith    Darren
1     Massage Room 1      4      1              Smith    Darren
2      Snooker Table      7      1              Smith    Darren
3         Pool Table      8      1              Smith    Darren
4         Pool Table      8      1              Smith    Darren
...              ...    ...    ...                ...       ...
3155      Pool Table      8     29  Worthington-Smyth     Henry
3156      Pool Table      8     29  Worthington-Smyth     Henry
3157      Pool Table      8     21          Mackenzie      Anna
3158      Pool Table      8     16              Baker   Timothy
3159      Pool Table      8     29  Worthington-Smyth     Henry

[3160 rows x 5 columns]


In [147]:
Q13_query = """WITH cte AS (SELECT Bookings.facid, Bookings.starttime FROM Bookings WHERE memid != 0)
SELECT Facilities.name, cte.facid, cte.starttime FROM cte JOIN Facilities ON cte.facid = Facilities.facid;"""


result_df = pd.read_sql(Q13_query, con = engine)
print(result_df)

                name  facid            starttime
0       Table Tennis      3  2012-07-03 11:00:00
1     Massage Room 1      4  2012-07-03 08:00:00
2      Snooker Table      7  2012-07-03 19:00:00
3         Pool Table      8  2012-07-03 10:00:00
4         Pool Table      8  2012-07-03 15:00:00
...              ...    ...                  ...
3155      Pool Table      8  2012-09-30 16:30:00
3156      Pool Table      8  2012-09-30 18:00:00
3157      Pool Table      8  2012-09-30 18:30:00
3158      Pool Table      8  2012-09-30 19:00:00
3159      Pool Table      8  2012-09-30 19:30:00

[3160 rows x 3 columns]
